In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import tensorflow as tf
import pandas as pd

2026-04-05 15:48:44.806844: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775404125.015583      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775404125.073795      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775404125.553729      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775404125.553775      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775404125.553777      55 computation_placer.cc:177] computation placer alr

In [3]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification

In [4]:
df = pd.DataFrame({
    "text": ["I love this movie", "Worst film ever", "Amazing experience", "Not good"],
    "label": [1, 0, 1, 0]
})

In [5]:
df

,text,label
0,I love this movie,1
1,Worst film ever,0
2,Amazing experience,1
3,Not good,0


In [6]:
tokenizer=RobertaTokenizer.from_pretrained('roberta-base')

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [7]:
encodings = tokenizer(
    df['text'].tolist(),   # convert to list
    padding=True,
    add_special_tokens=True,
    truncation=True,
    max_length=32,
    return_tensors="pt"
)

In [8]:
encodings

{'input_ids': tensor([[    0,   100,   657,    42,  1569,     2],
        [    0,   771, 36767,   822,   655,     2],
        [    0, 41710,   676,     2,     1,     1],
        [    0,  7199,   205,     2,     1,     1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 0, 0]])}

In [9]:
input_ids=encodings['input_ids']
input_ids

tensor([[    0,   100,   657,    42,  1569,     2],
        [    0,   771, 36767,   822,   655,     2],
        [    0, 41710,   676,     2,     1,     1],
        [    0,  7199,   205,     2,     1,     1]])

In [14]:
attention_mask=encodings['attention_mask']

In [10]:
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
import torch
optimizer = torch.optim.Adam(model.parameters(), lr=3e-5)
loss_fn = torch.nn.CrossEntropyLoss()

In [16]:
import torch

labels = torch.tensor(df['label'].values)

In [17]:
for epoch in range(3):
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits
    
    loss = loss_fn(logits, labels)              # Compares:predicted output (logits),actual labels
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

In [18]:
text = "This movie is amazing"

inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():                             # no training here saves memory
    outputs = model(**inputs)             #model(input_ids=inputs["input_ids"],attention_mask=inputs["attention_mask"])
    pred = torch.argmax(outputs.logits, dim=1)

print(pred)

tensor([1])


In [20]:
with torch.no_grad():
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits

    preds = torch.argmax(logits, dim=1)

    correct = (preds == labels).sum().item()
    accuracy = correct / len(labels)*100

    print("Accuracy:", accuracy)

Accuracy: 100.0
